In [1]:
%%writefile main.py
import math
import sys
import traceback
import time
from collections import defaultdict, namedtuple
from dataclasses import dataclass, field

# =============================================================================
# RL PARAMETER DICTIONARY (Optimized base weights)
# =============================================================================
PARAMS = {
    "ATTACK_COST_TURN_WEIGHT": 0.1954,
    "SNIPE_COST_TURN_WEIGHT": 0.2200,          
    "EARLY_NEUTRAL_VALUE_MULT": 4.1712,
    "OPENING_HOSTILE_TARGET_VALUE_MULT": 2.8002,
    "EARLY_SNIPE_VALUE_MULT": 3.2539,
    "FOUR_PLAYER_ROTATING_SEND_RATIO": 0.7935,
}

def update_params(new_weights_dict):
    global PARAMS
    for k, v in new_weights_dict.items():
        PARAMS[k] = v

# =============================================================================
# CONFIG & MULTIPLIERS
# =============================================================================

BOARD = 100.0
CENTER_X = 50.0
CENTER_Y = 50.0
SUN_R = 10.0
MAX_SPEED = 6.0
SUN_SAFETY = 1.5
ROTATION_LIMIT = 50.0
TOTAL_STEPS = 500
HORIZON = 110
LAUNCH_CLEARANCE = 0.1

EARLY_TURN_LIMIT = 40
OPENING_TURN_LIMIT = 80
LATE_REMAINING_TURNS = 70
VERY_LATE_REMAINING_TURNS = 25

SAFE_NEUTRAL_MARGIN = 2
CONTESTED_NEUTRAL_MARGIN = 2
INTERCEPT_TOLERANCE = 1

COMET_MAX_CHASE_TURNS = 10

INDIRECT_VALUE_SCALE = 0.15
INDIRECT_FRIENDLY_WEIGHT = 0.35
INDIRECT_NEUTRAL_WEIGHT = 0.9
INDIRECT_ENEMY_WEIGHT = 1.25

STATIC_NEUTRAL_VALUE_MULT = 1.0
ROTATING_OPENING_VALUE_MULT = 1.8
STATIC_HOSTILE_VALUE_MULT = 1.65
HOSTILE_TARGET_VALUE_MULT = 2.65

SAFE_NEUTRAL_VALUE_MULT = 1.2
CONTESTED_NEUTRAL_VALUE_MULT = 0.7
SNIPE_VALUE_MULT = 1.12
COMET_VALUE_MULT = 2.8 
SWARM_VALUE_MULT = 1.05
FINISHING_HOSTILE_VALUE_MULT = 1.15
BEHIND_ROTATING_NEUTRAL_VALUE_MULT = 0.92

NEUTRAL_MARGIN_BASE = 2
NEUTRAL_MARGIN_PROD_WEIGHT = 2
NEUTRAL_MARGIN_CAP = 8
HOSTILE_MARGIN_BASE = 3
HOSTILE_MARGIN_PROD_WEIGHT = 2
HOSTILE_MARGIN_CAP = 12
STATIC_TARGET_MARGIN = 4
CONTESTED_TARGET_MARGIN = 5
FOUR_PLAYER_TARGET_MARGIN = 3
LONG_TRAVEL_MARGIN_START = 18
LONG_TRAVEL_MARGIN_DIVISOR = 3
LONG_TRAVEL_MARGIN_CAP = 8
COMET_MARGIN_RELIEF = 6
FINISHING_HOSTILE_SEND_BONUS = 3

STATIC_TARGET_SCORE_MULT = 1.18
EARLY_STATIC_NEUTRAL_SCORE_MULT = 1.25
FOUR_PLAYER_ROTATING_NEUTRAL_SCORE_MULT = 0.84
DENSE_STATIC_NEUTRAL_COUNT = 4
DENSE_ROTATING_NEUTRAL_SCORE_MULT = 0.86

FOLLOWUP_MIN_SHIPS = 8
LOW_VALUE_COMET_PRODUCTION = 1
LATE_CAPTURE_BUFFER = 5
VERY_LATE_CAPTURE_BUFFER = 3

REAR_DISTANCE_RATIO = 1.25
REAR_STAGE_PROGRESS = 0.78
REAR_SEND_RATIO_TWO_PLAYER = 0.62
REAR_SEND_RATIO_FOUR_PLAYER = 0.7
REAR_SEND_MIN_SHIPS = 10
REAR_MAX_TRAVEL_TURNS = 40

PARTIAL_SOURCE_MIN_SHIPS = 6
MULTI_SOURCE_TOP_K = 5
MULTI_SOURCE_ETA_TOLERANCE = 2
MULTI_SOURCE_PLAN_PENALTY = 0.97
HOSTILE_SWARM_ETA_TOLERANCE = 1

THREE_SOURCE_SWARM_ENABLED = True
THREE_SOURCE_MIN_TARGET_SHIPS = 20
THREE_SOURCE_ETA_TOLERANCE = 1
THREE_SOURCE_PLAN_PENALTY = 0.93

REINFORCE_ENABLED = True
REINFORCE_MIN_PRODUCTION = 2
REINFORCE_MAX_TRAVEL_TURNS = 22
REINFORCE_SAFETY_MARGIN = 2
REINFORCE_VALUE_MULT = 1.35
REINFORCE_MAX_SOURCE_FRACTION = 0.75
REINFORCE_MIN_FUTURE_TURNS = 40

MULTI_ENEMY_PROACTIVE_HORIZON = 14
MULTI_ENEMY_PROACTIVE_RATIO = 0.22
MULTI_ENEMY_STACK_WINDOW = 3

PROACTIVE_DEFENSE_HORIZON = 12
PROACTIVE_DEFENSE_RATIO = 0.18

LATE_IMMEDIATE_SHIP_VALUE = 0.6
WEAK_ENEMY_THRESHOLD = 45
ELIMINATION_BONUS = 18.0

BEHIND_DOMINATION = -0.20
AHEAD_DOMINATION = 0.18
FINISHING_DOMINATION = 0.35
FINISHING_PROD_RATIO = 1.25
AHEAD_ATTACK_MARGIN_BONUS = 0.08
BEHIND_ATTACK_MARGIN_PENALTY = 0.05
FINISHING_ATTACK_MARGIN_BONUS = 0.08

DOOMED_EVAC_TURN_LIMIT = 24
DOOMED_MIN_SHIPS = 8

CRASH_EXPLOIT_ENABLED = True
CRASH_EXPLOIT_MIN_TOTAL_SHIPS = 10
CRASH_EXPLOIT_ETA_WINDOW = 2
CRASH_EXPLOIT_POST_CRASH_DELAY = 1

SOFT_ACT_DEADLINE = 0.82
HEAVY_PHASE_MIN_TIME = 0.16
HEAVY_ROUTE_PLANET_LIMIT = 32

# =============================================================================
# TYPES
# =============================================================================

Planet = namedtuple("Planet",["id", "owner", "x", "y", "radius", "ships", "production"])
Fleet = namedtuple("Fleet",["id", "owner", "x", "y", "angle", "from_planet_id", "ships"])

@dataclass(frozen=True)
class ShotOption:
    score: float
    src_id: int
    target_id: int
    angle: float
    turns: int
    needed: int
    send_cap: int
    mission: str = "capture"

@dataclass
class Mission:
    kind: str
    score: float
    target_id: int
    turns: int
    options: list = field(default_factory=list)

# =============================================================================
# ABSOLUTE PHYSICS ENGINE
# =============================================================================

def dist(ax, ay, bx, by): return math.hypot(ax - bx, ay - by)
def orbital_radius(planet): return dist(planet.x, planet.y, CENTER_X, CENTER_Y)
def is_static_planet(planet): return orbital_radius(planet) + planet.radius >= ROTATION_LIMIT

def fleet_speed(ships):
    if ships <= 1: return 1.0
    ratio = max(0.0, min(1.0, math.log(ships) / math.log(1000.0)))
    return 1.0 + (MAX_SPEED - 1.0) * (ratio ** 1.5)

def point_to_segment_distance(px, py, x1, y1, x2, y2):
    dx = x2 - x1; dy = y2 - y1
    seg_len_sq = dx * dx + dy * dy
    if seg_len_sq <= 1e-9: return dist(px, py, x1, y1)
    t = max(0.0, min(1.0, ((px - x1) * dx + (py - y1) * dy) / seg_len_sq))
    return dist(px, py, x1 + t * dx, y1 + t * dy)

def segment_hits_sun(x1, y1, x2, y2, safety=SUN_SAFETY):
    return point_to_segment_distance(CENTER_X, CENTER_Y, x1, y1, x2, y2) < SUN_R + safety

def launch_point(sx, sy, sr, angle):
    clearance = sr + LAUNCH_CLEARANCE
    return sx + math.cos(angle) * clearance, sy + math.sin(angle) * clearance

def actual_path_geometry(sx, sy, sr, tx, ty, tr):
    angle = math.atan2(ty - sy, tx - sx)
    start_x, start_y = launch_point(sx, sy, sr, angle)
    hit_distance = max(0.0, dist(sx, sy, tx, ty) - (sr + LAUNCH_CLEARANCE) - tr)
    end_x = start_x + math.cos(angle) * hit_distance
    end_y = start_y + math.sin(angle) * hit_distance
    return angle, start_x, start_y, end_x, end_y, hit_distance

def safe_angle_and_distance(sx, sy, sr, tx, ty, tr):
    angle, start_x, start_y, end_x, end_y, hit_distance = actual_path_geometry(sx, sy, sr, tx, ty, tr)
    if segment_hits_sun(start_x, start_y, end_x, end_y): return None
    return angle, hit_distance

def predict_planet_position(planet, initial_by_id, angular_velocity, current_step, turns):
    init = initial_by_id.get(planet.id)
    if init is None: return planet.x, planet.y
    r = dist(init.x, init.y, CENTER_X, CENTER_Y)
    if r + init.radius >= ROTATION_LIMIT: return planet.x, planet.y
    
    base_ang = math.atan2(init.y - CENTER_Y, init.x - CENTER_X)
    new_ang = base_ang + angular_velocity * (current_step + turns)
    return CENTER_X + r * math.cos(new_ang), CENTER_Y + r * math.sin(new_ang)

def predict_comet_position(planet_id, comets, turns):
    for group in comets:
        pids = group.get("planet_ids",[])
        if planet_id not in pids: continue
        idx = pids.index(planet_id)
        paths = group.get("paths",[])
        path_index = group.get("path_index", 0)
        if idx >= len(paths): return None
        future_idx = path_index + int(turns)
        if 0 <= future_idx < len(paths[idx]): return paths[idx][future_idx][0], paths[idx][future_idx][1]
        return None
    return None

def comet_remaining_life(planet_id, comets):
    for group in comets:
        pids = group.get("planet_ids",[])
        if planet_id not in pids: continue
        idx = pids.index(planet_id)
        paths = group.get("paths",[])
        path_index = group.get("path_index", 0)
        if idx < len(paths): return max(0, len(paths[idx]) - path_index)
    return 0

def estimate_arrival(sx, sy, sr, tx, ty, tr, ships):
    safe = safe_angle_and_distance(sx, sy, sr, tx, ty, tr)
    if safe is None: return None
    angle, total_d = safe
    turns = max(1, int(math.ceil(total_d / fleet_speed(max(1, ships)))))
    return angle, turns

def travel_time(sx, sy, sr, tx, ty, tr, ships):
    est = estimate_arrival(sx, sy, sr, tx, ty, tr, ships)
    if est is None: return 10 ** 9
    return est[1]

def predict_target_position(target, turns, initial_by_id, ang_vel, comets, comet_ids, current_step):
    if target.id in comet_ids: return predict_comet_position(target.id, comets, turns)
    return predict_planet_position(target, initial_by_id, ang_vel, current_step, turns)

def search_safe_intercept(src, target, ships, initial_by_id, ang_vel, comets, comet_ids, current_step):
    best, best_score, max_turns = None, None, HORIZON
    if target.id in comet_ids: max_turns = min(max_turns, max(0, comet_remaining_life(target.id, comets) - 1))

    for candidate_turns in range(1, max_turns + 1):
        pos = predict_target_position(target, candidate_turns, initial_by_id, ang_vel, comets, comet_ids, current_step)
        if pos is None: continue
        est = estimate_arrival(src.x, src.y, src.radius, pos[0], pos[1], target.radius, ships)
        if est is None: continue
        _, turns = est
        if abs(turns - candidate_turns) > INTERCEPT_TOLERANCE: continue

        actual_turns = max(turns, candidate_turns)
        actual_pos = predict_target_position(target, actual_turns, initial_by_id, ang_vel, comets, comet_ids, current_step)
        if actual_pos is None: continue

        confirm = estimate_arrival(src.x, src.y, src.radius, actual_pos[0], actual_pos[1], target.radius, ships)
        if confirm is None: continue

        if abs(confirm[1] - actual_turns) > INTERCEPT_TOLERANCE: continue

        score = (abs(confirm[1] - actual_turns), confirm[1], candidate_turns)
        if best is None or score < best_score:
            best_score = score
            best = (confirm[0], confirm[1], actual_pos[0], actual_pos[1])
    return best

def aim_with_prediction(src, target, ships, initial_by_id, ang_vel, comets, comet_ids, current_step):
    est = estimate_arrival(src.x, src.y, src.radius, target.x, target.y, target.radius, ships)
    if est is None: return search_safe_intercept(src, target, ships, initial_by_id, ang_vel, comets, comet_ids, current_step)

    tx, ty = target.x, target.y
    for _ in range(5):
        _, turns = est
        pos = predict_target_position(target, turns, initial_by_id, ang_vel, comets, comet_ids, current_step)
        if pos is None: return None
        ntx, nty = pos
        next_est = estimate_arrival(src.x, src.y, src.radius, ntx, nty, target.radius, ships)
        if next_est is None: return search_safe_intercept(src, target, ships, initial_by_id, ang_vel, comets, comet_ids, current_step)
        if abs(ntx - tx) < 0.3 and abs(nty - ty) < 0.3 and abs(next_est[1] - turns) <= INTERCEPT_TOLERANCE:
            return next_est[0], next_est[1], ntx, nty
        tx, ty, est = ntx, nty, next_est

    final_est = estimate_arrival(src.x, src.y, src.radius, tx, ty, target.radius, ships)
    if final_est is None: return search_safe_intercept(src, target, ships, initial_by_id, ang_vel, comets, comet_ids, current_step)
    return final_est[0], final_est[1], tx, ty

# =============================================================================
# WORLD MODEL
# =============================================================================

def fleet_target_planet(fleet, planets):
    best_planet, best_time = None, 1e9
    dir_x, dir_y = math.cos(fleet.angle), math.sin(fleet.angle)
    speed = fleet_speed(fleet.ships)

    for planet in planets:
        dx, dy = planet.x - fleet.x, planet.y - fleet.y
        proj = dx * dir_x + dy * dir_y
        if proj < 0: continue
        perp_sq = dx * dx + dy * dy - proj * proj
        if perp_sq >= planet.radius * planet.radius: continue
        hit_d = max(0.0, proj - math.sqrt(max(0.0, planet.radius * planet.radius - perp_sq)))
        turns = hit_d / speed
        if turns <= HORIZON and turns < best_time:
            best_time = turns
            best_planet = planet
    return best_planet, int(math.ceil(best_time)) if best_planet else None

def build_arrival_ledger(fleets, planets):
    arrivals_by_planet = {planet.id:[] for planet in planets}
    for fleet in fleets:
        target, eta = fleet_target_planet(fleet, planets)
        if target is not None:
            arrivals_by_planet[target.id].append((eta, fleet.owner, int(fleet.ships)))
    return arrivals_by_planet

def resolve_arrival_event(owner, garrison, arrivals):
    by_owner = {}
    for _, attacker_owner, ships in arrivals:
        by_owner[attacker_owner] = by_owner.get(attacker_owner, 0) + ships

    if not by_owner: return owner, max(0.0, garrison)
    sorted_players = sorted(by_owner.items(), key=lambda item: item[1], reverse=True)
    top_owner, top_ships = sorted_players[0]

    if len(sorted_players) > 1:
        if top_ships == sorted_players[1][1]: survivor_owner, survivor_ships = -1, 0
        else: survivor_owner, survivor_ships = top_owner, top_ships - sorted_players[1][1]
    else:
        survivor_owner, survivor_ships = top_owner, top_ships

    if survivor_ships <= 0: return owner, max(0.0, garrison)
    if owner == survivor_owner: return owner, garrison + survivor_ships
    garrison -= survivor_ships
    if garrison < 0: return survivor_owner, -garrison
    return owner, garrison

def simulate_planet_timeline(planet, arrivals, player, horizon):
    horizon = max(0, int(math.ceil(horizon)))
    by_turn = defaultdict(list)
    for turns, owner, ships in sorted([a for a in arrivals if a[2] > 0 and a[0] <= horizon], key=lambda x: x[0]):
        by_turn[max(1, int(math.ceil(turns)))].append((turns, owner, int(ships)))

    owner, garrison = planet.owner, float(planet.ships)
    owner_at, ships_at = {0: owner}, {0: max(0.0, garrison)}
    min_owned, first_enemy, fall_turn = garrison if owner == player else 0.0, None, None

    for turn in range(1, horizon + 1):
        if owner != -1: garrison += planet.production
        group = by_turn.get(turn,[])
        prev_owner = owner
        if group:
            if prev_owner == player and first_enemy is None and any(i[1] not in (-1, player) for i in group):
                first_enemy = turn
            owner, garrison = resolve_arrival_event(owner, garrison, group)
            if prev_owner == player and owner != player and fall_turn is None: fall_turn = turn

        owner_at[turn], ships_at[turn] = owner, max(0.0, garrison)
        if owner == player: min_owned = min(min_owned, garrison)

    keep_needed, holds_full = 0, True
    if planet.owner == player:
        def survives_with_keep(keep):
            sim_owner, sim_garrison = planet.owner, float(keep)
            for turn in range(1, horizon + 1):
                if sim_owner != -1: sim_garrison += planet.production
                if by_turn.get(turn,[]):
                    sim_owner, sim_garrison = resolve_arrival_event(sim_owner, sim_garrison, by_turn.get(turn,[]))
                    if sim_owner != player: return False
            return sim_owner == player

        if survives_with_keep(int(planet.ships)):
            lo, hi = 0, int(planet.ships)
            while lo < hi:
                mid = (lo + hi) // 2
                if survives_with_keep(mid): hi = mid
                else: lo = mid + 1
            keep_needed = lo
        else:
            holds_full, keep_needed = False, int(planet.ships)

    return {"owner_at": owner_at, "ships_at": ships_at, "keep_needed": keep_needed, "min_owned": max(0, int(math.floor(min_owned))) if planet.owner == player else 0, "first_enemy": first_enemy, "fall_turn": fall_turn, "holds_full": holds_full, "horizon": horizon}

def state_at_timeline(timeline, arrival_turn):
    turn = max(0, min(int(math.ceil(arrival_turn)), timeline["horizon"]))
    return timeline["owner_at"].get(turn, timeline["owner_at"][timeline["horizon"]]), max(0.0, timeline["ships_at"].get(turn, timeline["ships_at"][timeline["horizon"]]))

def count_players(planets, fleets):
    owners = set(p.owner for p in planets if p.owner != -1) | set(f.owner for f in fleets)
    return max(2, len(owners))

def nearest_distance_to_set(px, py, planets):
    return min((dist(px, py, p.x, p.y) for p in planets), default=10**9)

def indirect_wealth(planet, planets, player):
    wealth = 0.0
    for other in planets:
        if other.id == planet.id: continue
        d = dist(planet.x, planet.y, other.x, other.y)
        if d >= 1:
            factor = other.production / (d + 12.0)
            if other.owner == player: wealth += factor * INDIRECT_FRIENDLY_WEIGHT
            elif other.owner == -1: wealth += factor * INDIRECT_NEUTRAL_WEIGHT
            else: wealth += factor * INDIRECT_ENEMY_WEIGHT
    return wealth

def detect_enemy_crashes(arrivals_by_planet, player, eta_window):
    crashes =[]
    for target_id, arrivals in arrivals_by_planet.items():
        enemy_events = sorted([(eta, owner, ships) for eta, owner, ships in arrivals if owner not in (-1, player) and ships > 0], key=lambda x: x[0])
        if len(enemy_events) < 2: continue
        
        matched = False
        for i in range(len(enemy_events)):
            if matched: break
            for j in range(i + 1, len(enemy_events)):
                if enemy_events[i][1] == enemy_events[j][1]: continue
                if abs(enemy_events[i][0] - enemy_events[j][0]) > eta_window: continue
                total = enemy_events[i][2] + enemy_events[j][2]
                if total >= CRASH_EXPLOIT_MIN_TOTAL_SHIPS:
                    crashes.append({"target_id": target_id, "crash_turn": max(enemy_events[i][0], enemy_events[j][0]), "total_enemy_ships": total})
                    matched = True
                    break
    return crashes

class WorldModel:
    def __init__(self, player, step, planets, fleets, initial_by_id, ang_vel, comets, comet_ids):
        self.player, self.step = player, step
        self.planets, self.fleets = planets, fleets
        self.initial_by_id, self.ang_vel = initial_by_id, ang_vel
        self.comets, self.comet_ids = comets, set(comet_ids)

        self.planet_by_id = {p.id: p for p in planets}
        self.my_planets =[p for p in planets if p.owner == player]
        self.enemy_planets =[p for p in planets if p.owner not in (-1, player)]
        self.neutral_planets =[p for p in planets if p.owner == -1]
        self.static_neutral_planets =[p for p in self.neutral_planets if is_static_planet(p)]

        self.num_players = count_players(planets, fleets)
        self.remaining_steps = max(1, TOTAL_STEPS - step)
        self.is_early, self.is_opening = step < EARLY_TURN_LIMIT, step < OPENING_TURN_LIMIT
        self.is_late, self.is_very_late = self.remaining_steps < LATE_REMAINING_TURNS, self.remaining_steps < VERY_LATE_REMAINING_TURNS
        self.is_four_player = self.num_players >= 4

        self.cluster_score = {}
        for p in planets:
            self.cluster_score[p.id] = sum(o.production for o in planets if p.id != o.id and dist(p.x, p.y, o.x, o.y) < 25)

        self.owner_strength, self.owner_production = defaultdict(int), defaultdict(int)
        for p in planets:
            if p.owner != -1:
                self.owner_strength[p.owner] += int(p.ships)
                self.owner_production[p.owner] += int(p.production)
        for f in fleets: self.owner_strength[f.owner] += int(f.ships)

        self.my_total, self.my_prod = self.owner_strength.get(player, 0), self.owner_production.get(player, 0)
        self.enemy_total = sum(s for o, s in self.owner_strength.items() if o != player)
        self.max_enemy_strength = max((s for o, s in self.owner_strength.items() if o != player), default=0)
        self.enemy_prod = sum(p for o, p in self.owner_production.items() if o != player)

        # STANCE ENGINE
        incoming_to_us = sum(f.ships for f in fleets if f.owner != player and fleet_target_planet(f, planets)[0] and fleet_target_planet(f, planets)[0].owner == player)
        enemy_total_flying = sum(f.ships for f in fleets if f.owner != player)
        
        self.mode = "BALANCED"
        if self.is_early: self.mode = "OFFENSE"  
        elif incoming_to_us > self.my_prod * 2.5: self.mode = "DEFENSE"  
        elif self.my_prod > self.enemy_prod * 1.05 or (self.step > 150 and enemy_total_flying < self.enemy_total * 0.15): self.mode = "OFFENSE"  

        # DYNAMIC PLANET ROLES
        self.planet_roles = {}
        for p in self.my_planets:
            if p.production >= 4 and p.ships < 20: self.planet_roles[p.id] = "VULNERABLE_FACTORY"
            elif p.production <= 2 and p.ships >= 25: self.planet_roles[p.id] = "STAGING_BASE"
            elif p.production >= 3 and p.ships >= 40: self.planet_roles[p.id] = "JUGGERNAUT"
            else: self.planet_roles[p.id] = "OUTPOST"

        self.arrivals_by_planet = build_arrival_ledger(fleets, planets)
        self.base_timeline = {p.id: simulate_planet_timeline(p, self.arrivals_by_planet[p.id], player, HORIZON) for p in planets}
        self.indirect_wealth_map = {p.id: indirect_wealth(p, planets, player) for p in planets}
        self.reaction_cache, self.base_need_cache = {}, {}

        self.reserve, self.available, self.doomed_candidates, self.threatened_candidates = self._compute_defense_buffers()
        self.enemy_crashes = detect_enemy_crashes(self.arrivals_by_planet, player, CRASH_EXPLOIT_ETA_WINDOW) if CRASH_EXPLOIT_ENABLED and self.is_four_player else[]

    def _multi_enemy_proactive_keep(self, planet):
        if not self.enemy_planets: return 0
        
        threats = sorted([(travel_time(e.x, e.y, e.radius, planet.x, planet.y, planet.radius, max(1, e.ships)), int(e.ships)) for e in self.enemy_planets if travel_time(e.x, e.y, e.radius, planet.x, planet.y, planet.radius, max(1, e.ships)) <= MULTI_ENEMY_PROACTIVE_HORIZON], key=lambda x: x[0])
        if not threats: return 0

        best_stacked, left, running = 0, 0, 0
        for right in range(len(threats)):
            running += threats[right][1]
            while threats[right][0] - threats[left][0] > MULTI_ENEMY_STACK_WINDOW:
                running -= threats[left][1]
                left += 1
            best_stacked = max(best_stacked, running)

        p_ratio = MULTI_ENEMY_PROACTIVE_RATIO
        l_ratio = PROACTIVE_DEFENSE_RATIO
        if self.mode == "DEFENSE":
            p_ratio *= 1.5; l_ratio *= 1.5
        elif self.mode == "OFFENSE":
            p_ratio *= 0.2; l_ratio *= 0.2

        proactive = int(best_stacked * p_ratio)
        legacy = max((int(ships * l_ratio) for eta, ships in threats if eta <= PROACTIVE_DEFENSE_HORIZON), default=0)
        return max(proactive, legacy)

    def _compute_defense_buffers(self):
        reserve, available, doomed, threatened = {}, {}, set(), {}

        for planet in self.my_planets:
            timeline = self.base_timeline[planet.id]
            exact_keep = timeline["keep_needed"]
            proactive_keep = self._multi_enemy_proactive_keep(planet)

            role = self.planet_roles.get(planet.id, "OUTPOST")
            if role == "VULNERABLE_FACTORY": proactive_keep = max(proactive_keep, int(planet.production * 3))
            elif role == "STAGING_BASE" and self.mode == "OFFENSE": proactive_keep = 0 

            # ANTI-STARTER GARRISON
            empire_size = len(self.my_planets)
            if empire_size >= 4:
                mob_floor = int(planet.ships * 0.35) 
            else:
                mob_floor = max(2, int(planet.ships * 0.08)) 
            
            proactive_keep = max(proactive_keep, mob_floor)

            reserve[planet.id] = min(int(planet.ships), max(exact_keep, proactive_keep))
            available[planet.id] = max(0, int(planet.ships) - reserve[planet.id])

            if not timeline["holds_full"] and timeline["fall_turn"] is not None:
                ft = timeline["fall_turn"]
                if ft <= DOOMED_EVAC_TURN_LIMIT and planet.ships >= DOOMED_MIN_SHIPS: doomed.add(planet.id)
                if REINFORCE_ENABLED and planet.production >= REINFORCE_MIN_PRODUCTION and self.remaining_steps >= REINFORCE_MIN_FUTURE_TURNS:
                    dh = max((int(math.ceil(timeline["ships_at"].get(t, 0))) + 1 for t in range(1, ft + 1) if timeline["owner_at"].get(t) != self.player), default=0)
                    threatened[planet.id] = {"fall_turn": ft, "deficit_hint": max(1, dh)}
        return reserve, available, doomed, threatened

    def is_static(self, planet_id): return is_static_planet(self.planet_by_id[planet_id])
    def comet_life(self, planet_id): return comet_remaining_life(planet_id, self.comets)
    def source_inventory_left(self, source_id, spent_total): return max(0, int(self.planet_by_id[source_id].ships) - spent_total[source_id])
    def source_attack_left(self, source_id, spent_total): return max(0, self.available.get(source_id, 0) - spent_total[source_id])
    def plan_shot(self, src_id, target_id, ships): return aim_with_prediction(self.planet_by_id[src_id], self.planet_by_id[target_id], ships, self.initial_by_id, self.ang_vel, self.comets, self.comet_ids, self.step)
    
    def reaction_times(self, target_id):
        if target_id in self.reaction_cache: return self.reaction_cache[target_id]
        t = self.planet_by_id[target_id]
        my_t = min((travel_time(p.x, p.y, p.radius, t.x, t.y, t.radius, max(1, p.ships)) for p in self.my_planets), default=10**9)
        en_t = min((travel_time(p.x, p.y, p.radius, t.x, t.y, t.radius, max(1, p.ships)) for p in self.enemy_planets), default=10**9)
        self.reaction_cache[target_id] = (my_t, en_t)
        return my_t, en_t

    def projected_state(self, target_id, arrival_turn, planned_commitments=None, extra_arrivals=()):
        cutoff = max(1, int(math.ceil(arrival_turn)))
        arr =[i for i in self.arrivals_by_planet.get(target_id,[]) + (planned_commitments or {}).get(target_id,[]) + list(extra_arrivals) if i[0] <= cutoff]
        if not arr: return state_at_timeline(self.base_timeline[target_id], cutoff)
        return state_at_timeline(simulate_planet_timeline(self.planet_by_id[target_id], arr, self.player, cutoff), cutoff)

    def ships_needed_to_capture(self, target_id, arrival_turn, planned_commitments=None, extra_arrivals=()):
        cutoff = max(1, int(math.ceil(arrival_turn)))
        ckey = (target_id, cutoff) if not (planned_commitments or {}).get(target_id) and not extra_arrivals else None
        if ckey and ckey in self.base_need_cache: return self.base_need_cache[ckey]
        o_t, s_t = self.projected_state(target_id, cutoff, planned_commitments, extra_arrivals)
        need = 0 if o_t == self.player else int(math.ceil(s_t)) + 1
        if ckey: self.base_need_cache[ckey] = need
        return need

    def reinforcement_needed_for(self, planet_id, arrival_turn, planned_commitments=None):
        at = max(1, int(math.ceil(arrival_turn)))
        if self.planet_by_id[planet_id].owner != self.player: return self.ships_needed_to_capture(planet_id, at, planned_commitments)
        arr = self.arrivals_by_planet.get(planet_id,[]) + (planned_commitments or {}).get(planet_id,[])
        hz = max(at + 5, self.base_timeline[planet_id]["horizon"])
        tl = simulate_planet_timeline(self.planet_by_id[planet_id], arr, self.player, hz)
        return max((int(math.ceil(tl["ships_at"].get(t, 0))) + 1 for t in range(at, min(hz, at + 20) + 1) if tl["owner_at"].get(t) != self.player), default=0)

# =============================================================================
# STRATEGY & MISSIONS
# =============================================================================

def planet_distance(first, second): return math.hypot(first.x - second.x, first.y - second.y)

def build_modes(world):
    dom = (world.my_total - world.enemy_total) / max(1, world.my_total + world.enemy_total)
    ia, ib = dom > AHEAD_DOMINATION, dom < BEHIND_DOMINATION
    isf = dom > FINISHING_DOMINATION and world.my_prod > world.enemy_prod * FINISHING_PROD_RATIO and world.step > 100
    
    amm = 1.0 + (AHEAD_ATTACK_MARGIN_BONUS if ia else 0) - (BEHIND_ATTACK_MARGIN_PENALTY if ib else 0) + (FINISHING_ATTACK_MARGIN_BONUS if isf else 0)
    
    if world.mode == "DEFENSE": amm += 0.20  
    elif world.mode == "OFFENSE": amm -= 0.15  

    return {"domination": dom, "is_behind": ib, "is_ahead": ia, "is_dominating": ia or (world.max_enemy_strength > 0 and world.my_total > world.max_enemy_strength * 1.25), "is_finishing": isf, "attack_margin_mult": amm}

def is_safe_neutral(target, world): return target.owner == -1 and world.reaction_times(target.id)[0] <= world.reaction_times(target.id)[1] - SAFE_NEUTRAL_MARGIN
def is_contested_neutral(target, world): return target.owner == -1 and abs(world.reaction_times(target.id)[0] - world.reaction_times(target.id)[1]) <= CONTESTED_NEUTRAL_MARGIN

def opening_filter(target, arrival_turns, needed, src_available, world):
    if not world.is_opening or target.owner != -1 or target.id in world.comet_ids: return False
    
    if world.step <= 15 and needed <= src_available and arrival_turns <= 18: return False
    if world.is_early and needed <= src_available and needed <= 6: return False
    if world.is_static(target.id): return False

    my_t, enemy_t = world.reaction_times(target.id)
    if target.production >= SAFE_OPENING_PROD_THRESHOLD and arrival_turns <= SAFE_OPENING_TURN_LIMIT and enemy_t - my_t >= SAFE_NEUTRAL_MARGIN: return False
    if world.is_four_player: return not (needed <= max(PARTIAL_SOURCE_MIN_SHIPS, int(src_available * PARAMS["FOUR_PLAYER_ROTATING_SEND_RATIO"])) and arrival_turns <= FOUR_PLAYER_ROTATING_TURN_LIMIT and enemy_t - my_t >= FOUR_PLAYER_ROTATING_REACTION_GAP)
    return arrival_turns > ROTATING_OPENING_MAX_TURNS or target.production <= ROTATING_OPENING_LOW_PROD

def target_value(target, arrival_turns, mission, world, modes, src=None):
    tp = max(1, world.remaining_steps - arrival_turns)
    if target.id in world.comet_ids:
        tp = max(0, min(tp, world.comet_life(target.id) - arrival_turns))
        if tp <= 0: return -1.0

    production_value = target.production ** 1.6
    value = production_value * tp + world.indirect_wealth_map[target.id] * tp * INDIRECT_VALUE_SCALE
    value += (world.cluster_score.get(target.id, 0) * 0.2) * tp 

    if world.is_static(target.id): value *= STATIC_NEUTRAL_VALUE_MULT if target.owner == -1 else STATIC_HOSTILE_VALUE_MULT
    else: value *= ROTATING_OPENING_VALUE_MULT if world.is_opening else 1.0

    if target.owner not in (-1, world.player):
        value *= PARAMS["OPENING_HOSTILE_TARGET_VALUE_MULT"] if world.is_opening else HOSTILE_TARGET_VALUE_MULT
        if world.mode == "OFFENSE": value *= 1.35
    elif target.owner == -1:
        if is_safe_neutral(target, world): value *= SAFE_NEUTRAL_VALUE_MULT
        elif is_contested_neutral(target, world): value *= CONTESTED_NEUTRAL_VALUE_MULT
        if world.is_early: value *= PARAMS["EARLY_NEUTRAL_VALUE_MULT"]
        if world.mode == "DEFENSE" and arrival_turns > 15: value *= 0.5 

    if target.id in world.comet_ids: value *= COMET_VALUE_MULT
    if mission == "snipe": value *= (PARAMS["EARLY_SNIPE_VALUE_MULT"] if world.is_early else SNIPE_VALUE_MULT)
    elif mission == "swarm": value *= (SWARM_VALUE_MULT * (1.2 if world.mode == "OFFENSE" else 1.0)) 
    elif mission == "reinforce": value *= REINFORCE_VALUE_MULT
    elif mission == "crash_exploit": value *= CRASH_EXPLOIT_VALUE_MULT

    if world.is_late:
        value += max(0, target.ships) * LATE_IMMEDIATE_SHIP_VALUE
        if target.owner not in (-1, world.player) and world.owner_strength.get(target.owner, 0) <= WEAK_ENEMY_THRESHOLD: value += ELIMINATION_BONUS

    if modes["is_finishing"] and target.owner not in (-1, world.player): value *= FINISHING_HOSTILE_VALUE_MULT
    if modes["is_behind"] and target.owner == -1 and not world.is_static(target.id): value *= BEHIND_ROTATING_NEUTRAL_VALUE_MULT
    if modes["is_behind"] and target.owner == -1 and is_safe_neutral(target, world): value *= 1.08
    if modes["is_dominating"] and target.owner == -1 and is_contested_neutral(target, world): value *= 0.92

    if src:
        current_dist = dist(src.x, src.y, target.x, target.y)
        future_pos = predict_target_position(target, arrival_turns, world.initial_by_id, world.ang_vel, world.comets, world.comet_ids, world.step)
        if future_pos:
            future_dist = dist(src.x, src.y, future_pos[0], future_pos[1])
            if future_dist < current_dist - 2.0:
                value *= 1.5  
            elif future_dist > current_dist + 2.0:
                value *= 0.8  

        if current_dist > 60.0 and target.owner != -1 and world.mode != "DEFENSE": value *= 0.15 
        elif current_dist <= 55.0: value *= 1.35 
            
        if arrival_turns * 4.5 < current_dist: value *= 1.4 
        if world.step <= 15 and current_dist < 25: value *= 2.0 

    flutter = ((target.id * 73 + world.step) % 11) / 100.0
    value *= (1.0 + flutter)

    return value

def preferred_send(target, base_needed, arrival_turns, src_available, world, modes):
    if world.is_early and target.owner == -1: 
        return min(src_available, max(base_needed + 1, 3)) 

    send = max(base_needed, int(math.ceil(base_needed * modes["attack_margin_mult"])))
    margin = min(NEUTRAL_MARGIN_CAP if target.owner == -1 else HOSTILE_MARGIN_CAP, (NEUTRAL_MARGIN_BASE + target.production * NEUTRAL_MARGIN_PROD_WEIGHT if target.owner == -1 else HOSTILE_MARGIN_BASE + target.production * HOSTILE_MARGIN_PROD_WEIGHT))
    
    if world.is_static(target.id): margin += STATIC_TARGET_MARGIN
    if is_contested_neutral(target, world): margin += CONTESTED_TARGET_MARGIN
    if world.is_four_player: margin += FOUR_PLAYER_TARGET_MARGIN
    if arrival_turns > LONG_TRAVEL_MARGIN_START: margin += min(LONG_TRAVEL_MARGIN_CAP, arrival_turns // LONG_TRAVEL_MARGIN_DIVISOR)
    if target.id in world.comet_ids: margin = max(0, margin - COMET_MARGIN_RELIEF)
    if modes["is_finishing"] and target.owner not in (-1, world.player): margin += FINISHING_HOSTILE_SEND_BONUS
    
    if world.mode == "OFFENSE" and target.owner != -1: margin = max(0, margin - 3)
        
    return min(src_available, send + margin)

def apply_score_modifiers(score, target, mission, world):
    if world.is_static(target.id): score *= STATIC_TARGET_SCORE_MULT
    if world.is_early and target.owner == -1 and world.is_static(target.id): score *= EARLY_STATIC_NEUTRAL_SCORE_MULT
    if world.is_four_player and target.owner == -1 and not world.is_static(target.id): score *= FOUR_PLAYER_ROTATING_NEUTRAL_SCORE_MULT
    if len(world.static_neutral_planets) >= DENSE_STATIC_NEUTRAL_COUNT and target.owner == -1 and not world.is_static(target.id): score *= DENSE_ROTATING_NEUTRAL_SCORE_MULT
    if mission == "snipe": score *= SNIPE_SCORE_MULT
    elif mission == "swarm": score *= SWARM_SCORE_MULT
    return score

def build_raid_missions(world, planned_commitments, modes, src_budget_fn):
    if world.mode == "DEFENSE": return[]
    missions =[]
    raid_mult = 2.0 if world.mode == "OFFENSE" else 1.0
    
    for target in world.enemy_planets:
        if target.production < 3: continue 
        for src in world.my_planets:
            available = src_budget_fn(src.id)
            if available < 5: continue
            
            aim = world.plan_shot(src.id, target.id, max(1, target.ships))
            if not aim or aim[1] > 30: continue 
            
            o_at, s_at = world.projected_state(target.id, aim[1], planned_commitments)
            if o_at == world.player: continue
            
            raid_cost = int(math.ceil(s_at)) + 1
            if raid_cost <= 0 or raid_cost > available or raid_cost > target.production * 5: continue
            
            value = target.production * 15 * raid_mult 
            score = value / (raid_cost + aim[1] * PARAMS["ATTACK_COST_TURN_WEIGHT"] + 1.0)
            
            opt = ShotOption(score=score, src_id=src.id, target_id=target.id, angle=aim[0], turns=aim[1], needed=raid_cost, send_cap=raid_cost, mission="raid")
            missions.append(Mission(kind="raid", score=score, target_id=target.id, turns=aim[1], options=[opt]))
            
    return missions

def build_snipe_mission(src, target, src_available, world, planned_commitments, modes):
    if target.owner != -1: return None
    enemy_etas = sorted({int(math.ceil(eta)) for eta, owner, ships in world.arrivals_by_planet.get(target.id,[]) if owner not in (-1, world.player) and ships > 0})
    if not enemy_etas: return None

    rough = world.plan_shot(src.id, target.id, min(src_available, max(PARTIAL_SOURCE_MIN_SHIPS, int(target.ships) + 8)))
    if not rough: return None

    for enemy_eta in enemy_etas[:3]:
        if rough[1] > enemy_eta + 3: continue 
        sync_turn = max(rough[1], enemy_eta)
        if target.id in world.comet_ids and (sync_turn >= world.comet_life(target.id) or sync_turn > COMET_MAX_CHASE_TURNS): continue

        need = world.ships_needed_to_capture(target.id, sync_turn, planned_commitments)
        if need <= 0 or need > src_available: continue
        final = world.plan_shot(src.id, target.id, need)
        if not final or final[1] > enemy_eta + 2: continue

        sync_turn = max(final[1], enemy_eta)
        need = world.ships_needed_to_capture(target.id, sync_turn, planned_commitments)
        if need <= 0 or need > src_available: continue
        value = target_value(target, sync_turn, "snipe", world, modes, src=src)
        if value <= 0: continue

        score = apply_score_modifiers(value / (need + sync_turn * PARAMS["SNIPE_COST_TURN_WEIGHT"] + 1.0), target, "snipe", world)
        opt = ShotOption(score=score, src_id=src.id, target_id=target.id, angle=final[0], turns=final[1], needed=need, send_cap=need, mission="snipe")
        return Mission(kind="snipe", score=score, target_id=target.id, turns=sync_turn, options=[opt])
    return None

def build_reinforcement_missions(world, planned_commitments, modes, source_budget_fn):
    if not REINFORCE_ENABLED or not world.threatened_candidates: return[]
    missions =[]
    for target_id, info in world.threatened_candidates.items():
        target = world.planet_by_id[target_id]
        if not info["fall_turn"] or info["fall_turn"] > REINFORCE_MAX_TRAVEL_TURNS + 5: continue
        best = None
        for src in world.my_planets:
            if src.id == target_id: continue
            budget = source_budget_fn(src.id)
            cap = min(budget, int(src.ships * REINFORCE_MAX_SOURCE_FRACTION))
            if cap <= 0: continue
            aim = world.plan_shot(src.id, target.id, min(cap, max(PARTIAL_SOURCE_MIN_SHIPS, int(info["deficit_hint"]) + REINFORCE_SAFETY_MARGIN)))
            if not aim or aim[1] > REINFORCE_MAX_TRAVEL_TURNS or aim[1] > info["fall_turn"]: continue
            need = world.reinforcement_needed_for(target_id, aim[1], planned_commitments)
            if need <= 0 or min(cap, need + REINFORCE_SAFETY_MARGIN) < need: continue
            final = world.plan_shot(src.id, target.id, min(cap, need + REINFORCE_SAFETY_MARGIN))
            if not final or final[1] > info["fall_turn"]: continue
            if (v := target_value(target, final[1], "reinforce", world, modes, src=src)) > 0:
                if world.mode == "DEFENSE": v *= 1.5 
                sc = v / (min(cap, need + REINFORCE_SAFETY_MARGIN) + final[1] * 0.35 + 1.0)
                m = Mission("reinforce", sc, target_id, final[1],[ShotOption(sc, src.id, target_id, final[0], final[1], need, min(cap, need + REINFORCE_SAFETY_MARGIN), "reinforce")])
                if not best or m.score > best.score: best = m
        if best: missions.append(best)
    return missions

def build_crash_exploit_missions(world, planned_commitments, modes):
    if not CRASH_EXPLOIT_ENABLED or not world.enemy_crashes: return []
    missions =[]
    for crash in world.enemy_crashes:
        target = world.planet_by_id[crash["target_id"]]
        if target.owner == world.player: continue
        da = crash["crash_turn"] + CRASH_EXPLOIT_POST_CRASH_DELAY
        best = None
        for src in world.my_planets:
            if (p := min(int(src.ships), max(PARTIAL_SOURCE_MIN_SHIPS, 12))) <= 0: continue
            if not (aim := world.plan_shot(src.id, target.id, p)) or abs(aim[1] - da) > 2: continue
            if (nd := world.ships_needed_to_capture(target.id, aim[1], planned_commitments)) <= 0 or nd > int(src.ships): continue
            if not (f := world.plan_shot(src.id, target.id, nd)) or abs(f[1] - da) > 2 or world.ships_needed_to_capture(target.id, f[1], planned_commitments) <= 0: continue
            
            # --- THE WALL STREET FIX: Strict ROI Check for Comets/Late-Game ---
            lifespan = world.remaining_steps - f[1]
            if target.id in world.comet_ids:
                lifespan = min(lifespan, world.comet_life(target.id) - f[1])
                if lifespan * target.production < nd: continue # Toxic Asset
            elif world.is_late and target.owner == -1:
                if lifespan * target.production < nd: continue
            # -------------------------------------------------------------------
            
            if (v := target_value(target, f[1], "crash_exploit", world, modes, src=src)) > 0:
                sc = v / (nd + f[1] * PARAMS["SNIPE_COST_TURN_WEIGHT"] + 1.0)
                m = Mission("crash_exploit", sc, target.id, f[1],[ShotOption(sc, src.id, target.id, f[0], f[1], nd, nd, "crash_exploit")])
                if not best or m.score > best.score: best = m
        if best: missions.append(best)
    return missions

def plan_moves(world, deadline=None):
    modes = build_modes(world)
    planned_commitments = defaultdict(list)
    source_options_by_target = defaultdict(list)
    missions =[]
    moves =[]
    spent_total = defaultdict(int)

    def source_inventory_left(sid): return world.source_inventory_left(sid, spent_total)
    def source_attack_left(sid): return world.source_attack_left(sid, spent_total)
    def addm(sid, ang, s):
        snd = min(int(s), source_inventory_left(sid))
        if snd >= 1: moves.append([sid, float(ang), int(snd)]); spent_total[sid] += snd
        return snd

    missions.extend(build_reinforcement_missions(world, planned_commitments, modes, source_inventory_left))
    missions.extend(build_raid_missions(world, planned_commitments, modes, source_attack_left))

    for src in world.my_planets:
        if (sa := source_attack_left(src.id)) <= 0: continue
        for target in world.planets:
            if target.id == src.id or target.owner == world.player: continue

            routing_ships = min(sa, max(3, int(target.ships) + 2))
            r_aim = world.plan_shot(src.id, target.id, routing_ships)
            
            if not r_aim or (world.is_very_late and r_aim[1] > world.remaining_steps - VERY_LATE_CAPTURE_BUFFER): continue
            if target.id in world.comet_ids and (r_aim[1] >= world.comet_life(target.id) or r_aim[1] > COMET_MAX_CHASE_TURNS): continue
            r_need = world.ships_needed_to_capture(target.id, r_aim[1], planned_commitments)
            if r_need <= 0 or opening_filter(target, r_aim[1], r_need, sa, world): continue

            aim = world.plan_shot(src.id, target.id, max(1, preferred_send(target, r_need, r_aim[1], sa, world, modes)))
            if not aim or (world.is_very_late and aim[1] > world.remaining_steps - VERY_LATE_CAPTURE_BUFFER): continue
            if target.id in world.comet_ids and (aim[1] >= world.comet_life(target.id) or aim[1] > COMET_MAX_CHASE_TURNS): continue
            need = world.ships_needed_to_capture(target.id, aim[1], planned_commitments)
            if need <= 0 or opening_filter(target, aim[1], need, sa, world): continue

            # --- THE WALL STREET FIX: Strict ROI Check ---
            lifespan = world.remaining_steps - aim[1]
            if target.id in world.comet_ids:
                lifespan = min(lifespan, world.comet_life(target.id) - aim[1])
                if lifespan * target.production < need: continue # Toxic Asset
            elif target.owner == -1 and world.is_late:
                if lifespan * target.production < need: continue # Bad late-game investment
            # ----------------------------------------------

            scap = min(sa, preferred_send(target, need, aim[1], sa, world, modes))
            if scap < 1 or (scap < need and scap < PARTIAL_SOURCE_MIN_SHIPS) or (v := target_value(target, aim[1], "capture", world, modes, src=src)) <= 0: continue

            sc = apply_score_modifiers(v / (max(need, min(scap, preferred_send(target, need, aim[1], scap, world, modes))) + aim[1] * PARAMS["ATTACK_COST_TURN_WEIGHT"] + 1.0), target, "capture", world)
            opt = ShotOption(sc, src.id, target.id, aim[0], aim[1], need, scap, "capture")
            source_options_by_target[target.id].append(opt)

            if scap >= need: missions.append(Mission("single", sc, target.id, aim[1], [opt]))
            if (sn := build_snipe_mission(src, target, sa, world, planned_commitments, modes)): missions.append(sn)

    for target_id, options in source_options_by_target.items():
        if len(options) < 2: continue
        target = world.planet_by_id[target_id]
        top = sorted(options, key=lambda i: -i.score)[:MULTI_SOURCE_TOP_K]
        ht = target.owner not in (-1, world.player)
        et2 = HOSTILE_SWARM_ETA_TOLERANCE if ht else MULTI_SOURCE_ETA_TOLERANCE

        for i in range(len(top)):
            for j in range(i + 1, len(top)):
                if top[i].src_id == top[j].src_id or abs(top[i].turns - top[j].turns) > et2: continue
                jt = max(top[i].turns, top[j].turns)
                nd = world.ships_needed_to_capture(target_id, jt, planned_commitments)
                if nd > 0 and top[i].send_cap < nd and top[j].send_cap < nd and top[i].send_cap + top[j].send_cap >= nd and (v := target_value(target, jt, "swarm", world, modes)) > 0:
                    missions.append(Mission("swarm", apply_score_modifiers(v / (nd + jt * PARAMS["ATTACK_COST_TURN_WEIGHT"] + 1.0), target, "swarm", world) * MULTI_SOURCE_PLAN_PENALTY, target_id, jt, [top[i], top[j]]))

        if THREE_SOURCE_SWARM_ENABLED and ht and int(target.ships) >= THREE_SOURCE_MIN_TARGET_SHIPS and len(top) >= 3:
            for i in range(len(top)):
                for j in range(i + 1, len(top)):
                    for k in range(j + 1, len(top)):
                        tr = [top[i], top[j], top[k]]
                        if len({o.src_id for o in tr}) < 3 or max(o.turns for o in tr) - min(o.turns for o in tr) > THREE_SOURCE_ETA_TOLERANCE: continue
                        jt = max(o.turns for o in tr)
                        nd = world.ships_needed_to_capture(target_id, jt, planned_commitments)
                        if nd > 0 and sum(o.send_cap for o in tr) >= nd and not any(tr[a].send_cap + tr[b].send_cap >= nd for a in range(3) for b in range(a + 1, 3)) and (v := target_value(target, jt, "swarm", world, modes)) > 0:
                            missions.append(Mission("swarm", apply_score_modifiers(v / (nd + jt * PARAMS["ATTACK_COST_TURN_WEIGHT"] + 1.0), target, "swarm", world) * THREE_SOURCE_PLAN_PENALTY, target_id, jt, tr))

    missions.sort(key=lambda i: -i.score)

    for m in missions:
        tgt = world.planet_by_id[m.target_id]
        if m.kind in ("single", "snipe", "reinforce", "crash_exploit", "raid"):
            o = m.options[0]
            left = source_inventory_left(o.src_id) if m.kind == "reinforce" else source_attack_left(o.src_id)
            if left <= 0: continue
            miss = world.reinforcement_needed_for(o.target_id, o.turns, planned_commitments) if m.kind == "reinforce" else (o.needed if m.kind == "raid" else world.ships_needed_to_capture(tgt.id, o.turns, planned_commitments))
            if miss <= 0 or min(left, o.send_cap) < miss: continue
            
            s_val = miss if m.kind in ("snipe", "crash_exploit", "raid") else min(min(left, o.send_cap), miss + REINFORCE_SAFETY_MARGIN if m.kind == "reinforce" else max(miss, preferred_send(tgt, miss, o.turns, min(left, o.send_cap), world, modes)))
            if s_val < miss: continue
            if (st := addm(o.src_id, o.angle, s_val)) >= miss: planned_commitments[tgt.id].append((o.turns, world.player, int(st)))
            continue

        lims =[min(source_attack_left(o.src_id), o.send_cap) for o in m.options]
        if min(lims) <= 0 or (miss := world.ships_needed_to_capture(tgt.id, m.turns, planned_commitments)) <= 0 or sum(lims) < miss: continue
        
        ord_o = sorted(zip(m.options, lims), key=lambda x: (x[0].turns, -x[1], x[0].src_id))
        rm, snds = miss, {}
        for idx, (o, l) in enumerate(ord_o):
            s = min(l, max(0, rm - sum(ol for _, ol in ord_o[idx + 1:])))
            snds[o.src_id], rm = s, rm - s
        if rm > 0: continue

        com =[]
        for o, _ in ord_o:
            if snds.get(o.src_id, 0) > 0 and (act := addm(o.src_id, o.angle, snds[o.src_id])) > 0: com.append((o.turns, world.player, int(act)))
        if sum(i[2] for i in com) >= miss: planned_commitments[tgt.id].extend(com)

    if not world.is_very_late:
        for src in world.my_planets:
            if (sl := source_attack_left(src.id)) < FOLLOWUP_MIN_SHIPS: continue
            bst = None
            for tgt in world.planets:
                if tgt.id == src.id or tgt.owner == world.player or (tgt.id in world.comet_ids and tgt.production <= LOW_VALUE_COMET_PRODUCTION): continue
                
                routing_ships = min(sl, max(3, int(tgt.ships) + 2))
                r_aim = world.plan_shot(src.id, tgt.id, routing_ships)
                if not r_aim or (world.is_late and r_aim[1] > world.remaining_steps - LATE_CAPTURE_BUFFER): continue
                rn = world.ships_needed_to_capture(tgt.id, r_aim[1], planned_commitments)
                if rn <= 0 or opening_filter(tgt, r_aim[1], rn, sl, world): continue
                
                # --- THE WALL STREET FIX ---
                lifespan = world.remaining_steps - r_aim[1]
                if tgt.owner == -1 and world.is_late and (lifespan * tgt.production) < rn: continue
                # -----------------------------

                snd = preferred_send(tgt, rn, r_aim[1], sl, world, modes)
                if snd < rn or (v := target_value(tgt, r_aim[1], "capture", world, modes, src=src)) <= 0: continue
                sc = apply_score_modifiers(v / (snd + r_aim[1] * PARAMS["ATTACK_COST_TURN_WEIGHT"] + 1.0), tgt, "capture", world)
                if not bst or sc > bst[0]: bst = (sc, tgt, snd)
            
            if bst and (aim := world.plan_shot(src.id, bst[1].id, bst[2])):
                miss = world.ships_needed_to_capture(bst[1].id, aim[1], planned_commitments)
                if miss > 0 and (s_val := min(source_attack_left(src.id), max(miss, preferred_send(bst[1], miss, aim[1], source_attack_left(src.id), world, modes)))) >= miss and (act := addm(src.id, aim[0], s_val)) >= miss:
                    planned_commitments[bst[1].id].append((aim[1], world.player, int(act)))

    if world.doomed_candidates:
        ft = world.enemy_planets if world.enemy_planets else (world.static_neutral_planets or world.neutral_planets)
        fd = {p.id: nearest_distance_to_set(p.x, p.y, ft) for p in world.my_planets} if ft else {p.id: 10**9 for p in world.my_planets}
        
        for p in world.my_planets:
            if p.id not in world.doomed_candidates or (planned_commitments.get(p.id) and sum(s for _, o, s in planned_commitments[p.id] if o == world.player) > 0) or (av := source_inventory_left(p.id)) < world.reserve.get(p.id, 0): continue
            
            bc = None
            for t in world.planets:
                if t.id == p.id or t.owner == world.player: continue
                pa = world.plan_shot(p.id, t.id, av)
                if not pa or pa[1] > world.remaining_steps - 2: continue
                nd = world.ships_needed_to_capture(t.id, pa[1], planned_commitments)
                if 0 < nd <= av and (fa := world.plan_shot(p.id, t.id, nd)):
                    sc = target_value(t, fa[1], "capture", world, modes, src=p) / (nd + fa[1] + 1.0) * (1.05 if t.owner not in (-1, world.player) else 1.0)
                    if not bc or sc > bc[0]: bc = (sc, t.id, fa[0], fa[1], nd)
            
            if bc and (act := addm(p.id, bc[2], bc[4])) >= 1: planned_commitments[bc[1]].append((bc[3], world.player, int(act)))
            elif (sa :=[a for a in world.my_planets if a.id != p.id and a.id not in world.doomed_candidates]):
                rt = min(sa, key=lambda a: (fd.get(a.id, 10**9), planet_distance(p, a)))
                if (aim := world.plan_shot(p.id, rt.id, av)): addm(p.id, aim[0], av)

    if (world.enemy_planets or world.neutral_planets) and len(world.my_planets) > 1 and not world.is_late:
        ft = world.enemy_planets if world.enemy_planets else (world.static_neutral_planets or world.neutral_planets)
        fd = {p.id: nearest_distance_to_set(p.x, p.y, ft) for p in world.my_planets}
        sf =[p for p in world.my_planets if p.id not in world.doomed_candidates]
        if sf:
            fa = min(sf, key=lambda p: fd[p.id])
            sr = REAR_SEND_RATIO_FOUR_PLAYER if world.is_four_player or modes["is_finishing"] else REAR_SEND_RATIO_TWO_PLAYER
            for r in sorted(world.my_planets, key=lambda p: -fd[p.id]):
                if r.id == fa.id or r.id in world.doomed_candidates or source_attack_left(r.id) < REAR_SOURCE_MIN_SHIPS or fd[r.id] < fd[fa.id] * REAR_DISTANCE_RATIO: continue
                sc =[p for p in sf if p.id != r.id and fd[p.id] < fd[r.id] * REAR_STAGE_PROGRESS]
                fr = min(sc, key=lambda p: planet_distance(r, p)) if sc else (min([p for p in sf if p.id != r.id], key=lambda p: planet_distance(p, min(ft, key=lambda t: planet_distance(r, t)))) if len(sf) > 1 else None)
                if fr and fr.id != r.id and (snd := int(source_attack_left(r.id) * sr)) >= REAR_SEND_MIN_SHIPS and (aim := world.plan_shot(r.id, fr.id, snd)) and aim[1] <= REAR_MAX_TRAVEL_TURNS: addm(r.id, aim[0], snd)

    fm, uf =[], defaultdict(int)
    for sid, a, s in moves:
        if (ms := min(s, int(world.planet_by_id[sid].ships) - uf[sid])) >= 1:
            fm.append([sid, float(a), int(ms)]); uf[sid] += ms
    return fm

# =============================================================================
# AGENT ENTRY POINT (Safely Wrapped)
# =============================================================================
_agent_step = 0

def _read(obs, key, default=None): 
    val = obs.get(key) if isinstance(obs, dict) else getattr(obs, key, None)
    return default if val is None else val

def build_world(obs):
    pl =[Planet(*p) for p in _read(obs, "planets", [])]
    fl =[Fleet(*f) for f in _read(obs, "fleets", [])]
    ip = {Planet(*p).id: Planet(*p) for p in _read(obs, "initial_planets",[])}
    return WorldModel(_read(obs, "player", 0), _read(obs, "step", 0) or 0, pl, fl, ip, _read(obs, "angular_velocity", 0.0) or 0.0, _read(obs, "comets", []) or[], set(_read(obs, "comet_planet_ids",[]) or[]))

def agent(obs, config=None):
    global _agent_step
    _agent_step += 1
    try:
        world = build_world(obs)
        if not world.my_planets: return[]
        
        act_timeout = 1.0
        if config is not None:
            act_timeout = _read(config, "actTimeout", 1.0)
            
        deadline = time.perf_counter() + min(SOFT_ACT_DEADLINE, max(0.55, act_timeout * 0.82))
        return plan_moves(world, deadline=deadline)
    except Exception:
        traceback.print_exc(file=sys.stderr)
        return []

__all__ =["agent"]

Writing main.py
